# Neural activity-trace data audit

This notebook visualizes the reproducible outputs from `scripts/audit_data.py`. The analysis is descriptive: correlations and principal components are not interpreted as causal connectivity.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path.cwd()
if ROOT.name == 'analysis' and ROOT.parent.name == 'agent_workspace':
    ROOT = ROOT.parent.parent
INPUT = ROOT / 'agent_workspace/logs/scripts/audit_data/initial'
OUTPUT = ROOT / 'figures/261027神经活动轨迹数据审计'
OUTPUT.mkdir(parents=True, exist_ok=True)
summary = json.loads((INPUT / 'summary.json').read_text())
acf = pd.read_csv(INPUT / 'lag_autocorrelation.csv')
psd = pd.read_csv(INPUT / 'state_power_spectra.csv')
state_summary = pd.read_csv(INPUT / 'sample_state_summary.csv')
pca = np.load(INPUT / 'sample_pca.npz', allow_pickle=True)
regions = np.load(INPUT / 'region_means.npz', allow_pickle=True)
summary

{'shape': [28801, 62130],
 'dtype': 'float32',
 'sampling_hz': 4.146313525439001,
 'duration_seconds': 6945.929154489284,
 'state_counts': {'Awake': 17105, 'NREM': 9622, 'REM': 2074},
 'n_segments': 98,
 'n_regions': 25,
 'region_counts': {'VISp1': 11052,
  'SSp-bfd1': 8572,
  'MOp1': 3693,
  'SSp-ul1': 3320,
  'SSs1': 3271,
  'SSp-n1': 3180,
  'SSp-m1': 3055,
  'RSPagl1': 2719,
  'RSPd1': 2328,
  'VISa1': 2240,
  'SSp-ll1': 2185,
  'VISpm1': 2080,
  'VISrl1': 1711,
  'AUDd1': 1706,
  'SSp-un1': 1490,
  'VISal1': 1447,
  'SSp-tr1': 1387,
  'VISl1': 1378,
  'VISam1': 1106,
  'TEa1': 937,
  'AUDp1': 897,
  'AUDpo1': 802,
  'VISli1': 739,
  'MOs1': 662,
  'VISpor1': 173},
 'full_value_summary': {'min': -39556.40625,
  'max': 403302.1875,
  'mean': 17647.002994268478,
  'std': 7545.735701965918,
  'nan': 0,
  'inf': 0,
  'count': 1789406130},
 'sample_nodes': 2048,
 'pca_nodes': 1024,
 'pca_explained_first': 0.3644466996192932,
 'pca_explained_first_5': 0.5360798239707947,
 'pca_explained_

In [2]:
colors = {'Awake': '#D55E00', 'NREM': '#0072B2', 'REM': '#009E73'}
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(np.asarray(axes).reshape(-1))
ax = next(ax_loader)
for state, group in acf.groupby('state'):
    ax.plot(group['seconds'], group['median'], marker='o', label=state, color=colors[state])
    ax.fill_between(group['seconds'], group['p10'], group['p90'], color=colors[state], alpha=0.15)
ax.axhline(0, color='black', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Lag (s)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Within-state memory')
ax.legend(frameon=False)
ax = next(ax_loader)
for state, group in psd.groupby('state'):
    group = group[group.frequency_hz > 0]
    ax.loglog(group.frequency_hz, group.mean_power, marker='o', ms=3, label=state, color=colors[state])
ax.axvline(2.0, color='black', ls='--', lw=0.8, label='Nyquist')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Normalized power')
ax.set_title('State spectra')
ax.legend(frameon=False)
for suffix in ('png', 'svg', 'pdf'):
    fig.savefig(OUTPUT / f'temporal_memory_and_spectra.{suffix}', bbox_inches='tight')
plt.show()

In [3]:
evr = pca['explained_variance_ratio']
scores = pca['scores']
labels = pca['labels'].astype(str)
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(np.asarray(axes).reshape(-1))
ax = next(ax_loader)
ax.plot(np.arange(1, len(evr) + 1), np.cumsum(evr), marker='o')
ax.axhline(0.5, color='black', ls='--', lw=0.8)
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative variance')
ax.set_title('PCA variance')
ax = next(ax_loader)
thin = np.arange(0, len(scores), 4)
for state in colors:
    mask = labels[thin] == state
    ax.scatter(scores[thin][mask, 0], scores[thin][mask, 1], s=3, alpha=0.25, color=colors[state], label=state)
ax.set_xlabel('PC1 score')
ax.set_ylabel('PC2 score')
ax.set_title('PC1–PC2 by state')
ax.legend(frameon=False, markerscale=3)
for suffix in ('png', 'svg', 'pdf'):
    fig.savefig(OUTPUT / f'low_rank_structure.{suffix}', bbox_inches='tight')
plt.show()

In [4]:
values = regions['values'].astype(np.float64)
region_names = regions['regions'].astype(str)
z = (values - values.mean(0)) / values.std(0)
corr = np.corrcoef(z.T)
fig, axes = get_fig(RN=1, CN=1, AW=7, AH=6, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(np.asarray(axes).reshape(-1))
ax = next(ax_loader)
image = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(np.arange(len(region_names)), region_names, rotation=90, fontsize=7)
ax.set_yticks(np.arange(len(region_names)), region_names, fontsize=7)
ax.set_title('Whole-recording regional functional correlation')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label='Pearson r')
for suffix in ('png', 'svg', 'pdf'):
    fig.savefig(OUTPUT / f'regional_functional_correlation.{suffix}', bbox_inches='tight')
plt.show()

## Interpretation

- PC1 explains about 36% of sampled-node variance and the first five PCs explain about 54%, supporting a low-rank common-drive component.
- Autocorrelation remains strongly positive for tens of seconds. This can reflect neural state, indicator kinetics, bleaching/baseline drift, or all of them.
- Spectral power is concentrated at the lowest resolved frequencies. Frequencies near the 2 Hz Nyquist limit should not be interpreted as faithful electrophysiological oscillations.
- Regional correlations are functional associations only. Common inputs and observation artifacts have not yet been removed.